# Kiwi 형태소 분석 — 1단계 (TF-IDF + 로지스틱 회귀용 토큰화, 2gram)
## 대상: 한국 올리브영

피드백 반영 사항
- ABSA 미적용, 분석 단위는 리뷰 1건 (제품_순위·리뷰_번호별 1행)
- `리뷰_불용어처리` 컬럼을 Kiwi로 형태소 분석 → 명사·형용사·동사만 남김
- 종속변수 `Y`: 별점 4·5 → 1, 1·2 → 0, 3점은 제외
- `리뷰_형태소` 토큰화 시 부정 표현(`~하지 않다`)을 분리하지 않고 하나의 토큰으로 결합, 1글자 키워드(`향` 등)도 삭제하지 않음

In [ ]:
!pip install kiwipiepy -q

In [ ]:
import pandas as pd
from kiwipiepy import Kiwi
from google.colab import files

uploaded = files.upload()  # 여기서 파일 선택창이 뜸
filename = list(uploaded.keys())[0]
df = pd.read_excel(filename)
print(df.shape)
df.head(2)

## 1. 종속변수 생성

이미 리뷰 1건 = 1행인 파일. 별점 4·5 → 1, 1·2 → 0, 3점은 제외.

In [ ]:
original_n = len(df)
df = df[df['별점'] != 3].copy()
df['Y'] = (df['별점'] >= 4).astype(int)

print(f'리뷰 건수: {len(df)} (3점 {original_n - len(df)}건 제외)')
print(df['Y'].value_counts())

## 2. 사용자 사전 등록

제품명·성분명은 형태소 분석기가 잘못 쪼갤 수 있으므로 미리 등록. 실제 데이터에 등장하는 제품명/성분명으로 채워 넣을 것.

In [ ]:
kiwi = Kiwi()

# TODO: 팀 데이터에 등장하는 제품명 / 성분명 리스트로 채우기
user_words = [
    # ('스킨푸드', 'NNP'),
    # ('히알루론산', 'NNG'),
]
for word, tag in user_words:
    kiwi.add_user_word(word, tag)

## 3. 형태소 분석 — 명사·형용사·동사·부정보조용언만 유지 (부정 표현 결합 버전)

- 유지 태그: `NNG`,`NNP`,`NNB`(일반/고유/의존 명사), `VV`(동사), `VA`(형용사), `XR`(어근), `VX`(보조용언 — `않다`·`못하다` 등 긴 부정)
  - kiwi는 불규칙 활용 어간에 `VA-I`, `VV-R`처럼 접미사를 붙이는 경우가 있어(예: `부드럽다` → `부드럽/VA-I`), 태그를 비교할 때 `-` 앞부분만 보도록 처리해 놓치지 않게 함
- **[수정] 1글자 이하 단어도 삭제하지 않음**
  - 기존에는 2글자 미만 명사를 노이즈(조사 잔여물 등)로 보고 제외했는데, 이 때문에 중요 키워드인 `향`(1글자)이 사라지는 문제가 있었음
  - 길이 제한을 없애고 태그 기준으로만 필터링하도록 수정
- **[수정] 부정 표현을 분리하지 않고 하나의 토큰으로 결합**
  - 기존 문제: `끈적이지 않아요`가 `끈적이다`, `않다` 두 개의 토큰으로 따로 분리되어 부정문인지 알아보기 어려웠음
  - 수정 후: `~하지 않다` / `~하지 못하다` 패턴을 감지해서 `끈적이지_않다`, `촉촉하지_않다`처럼 **하나의 토큰**으로 결합
    (명사+`하다` 형태인 `사용하지 못했어요` → `사용하지_못하다`, 불규칙 활용인 `부드럽지 않고` → `부드럽지_않다`도 동일하게 처리)
  - `안 끈적여요`처럼 부정 부사(`안`,`못`)가 용언 앞에 오는 경우도 `안_끈적이다`처럼 하나로 결합
  - 공백 대신 `_`로 이어붙여서, 토큰 문자열을 공백 기준으로 나눠 n-gram을 만들 때도 부정 표현이 하나의 단어로 유지되도록 함

In [ ]:
KEEP_TAGS = {'NNG', 'NNP', 'NNB', 'VV', 'VA', 'XR', 'VX'}
NEG_ADVERBS = {'안', '못'}          # MAG 중 부정 부사만 예외로 유지
NEG_AUX = {'않', '못하'}            # VX 중 부정 보조용언(않다/못하다)
PRED_SUFFIX_TAGS = {'XSA', 'XSV'}   # 명사를 용언으로 만드는 접미사 (예: 사용+하다, 실망+하다)


def _base_tag(tag: str) -> str:
    # kiwi는 불규칙 활용 어간에 VV-I / VA-R 처럼 접미사를 붙이므로, '-' 앞부분만 비교
    return tag.split('-')[0]


def _stem_span(raw, i):
    """i번째 토큰에서 시작하는 용언 어간의 (표면형, 다음 인덱스)를 반환. 해당 없으면 None."""
    tok = raw[i]
    base = _base_tag(tok.tag)
    n = len(raw)
    if base in ('VV', 'VA'):
        return tok.form, i + 1
    if base == 'XR':
        j = i + 1
        text = tok.form
        if j < n and raw[j].tag == 'XSA' and raw[j].form == '하':  # 촉촉+하다 등
            text += raw[j].form
            j += 1
        return text, j
    if base in ('NNG', 'NNP'):
        j = i + 1
        if j < n and raw[j].tag in PRED_SUFFIX_TAGS and raw[j].form == '하':  # 사용+하다 등
            return tok.form + raw[j].form, j + 1
    return None


def _match_negation(raw, start):
    """start 위치부터 '지(+조사)? + 않다/못하다' 부정 표현이 이어지는지 확인해
    (이어붙일 문자열, 다음 인덱스)를 반환. 아니면 None."""
    n = len(raw)
    j = start
    if not (j < n and raw[j].tag == 'EC' and raw[j].form == '지'):
        return None
    j += 1
    while j < n and raw[j].tag in ('JX', 'JKO', 'JKB', 'JKS'):  # '촉촉하진(=하지+ㄴ) 않아요' 같은 축약 처리
        j += 1
    if j < n and raw[j].tag == 'VX' and raw[j].form in NEG_AUX:
        return f'지_{raw[j].form}다', j + 1
    if j < n and raw[j].tag == 'MAG' and raw[j].form == '못' and j + 1 < n \
            and raw[j + 1].tag == 'VV' and raw[j + 1].form == '하':
        return '지_못하다', j + 2
    return None


def tokenize(text: str) -> str:
    if not isinstance(text, str) or not text.strip():
        return ''

    raw = list(kiwi.tokenize(text))
    tokens = []
    i, n = 0, len(raw)

    while i < n:
        tok = raw[i]
        base = _base_tag(tok.tag)

        # (1) 용언 어간 + '지 않다/못하다' -> "~하지 않다"를 하나의 토큰으로 결합
        stem = _stem_span(raw, i)
        if stem:
            stem_text, j = stem
            neg = _match_negation(raw, j)
            if neg:
                suffix, next_i = neg
                tokens.append(f'{stem_text}{suffix}')
                i = next_i
                continue

        # (2) 부정 부사(안/못) 바로 뒤에 용언이 오는 경우 -> 하나의 토큰으로 결합
        if tok.tag == 'MAG' and tok.form in NEG_ADVERBS:
            merged = None
            j = i + 1
            if j < n:
                nxt = raw[j]
                nbase = _base_tag(nxt.tag)
                if nbase in ('VV', 'VA'):
                    merged = (f'{tok.form}_{nxt.form}다', j + 1)
                elif nbase == 'XR':
                    k = j + 1
                    extra = ''
                    if k < n and raw[k].tag == 'XSA' and raw[k].form == '하':
                        extra = raw[k].form
                        k += 1
                    merged = (f'{tok.form}_{nxt.form}{extra}다', k)
            if merged:
                tokens.append(merged[0])
                i = merged[1]
                continue
            tokens.append(tok.form)  # 뒤에 용언이 없으면 부정 부사만 유지
            i += 1
            continue

        if base not in KEEP_TAGS:
            i += 1
            continue

        form = tok.form
        if base in ('VV', 'VA', 'VX'):
            form = form + '다'
        tokens.append(form)
        i += 1

    return ' '.join(tokens)

In [ ]:
tokenized = []
total = len(df)
for i, text in enumerate(df['리뷰_불용어처리'], start=1):
    tokenized.append(tokenize(text))
    if i % 200 == 0 or i == total:
        print(f'[{i}/{total}] 처리 중...')

df['리뷰_형태소'] = tokenized

In [ ]:
# 결과 확인
empty_count = (df['리뷰_형태소'].str.strip() == '').sum()
print(f'토큰화 후 빈 리뷰: {empty_count}건')
df[['리뷰_불용어처리', '리뷰_형태소']].head(5)

In [ ]:
df.to_excel('한국_올리브영_리뷰_형태소분석.xlsx', index=False)
print('저장 완료: 한국_올리브영_리뷰_형태소분석.xlsx')

from google.colab import files
files.download('한국_올리브영_리뷰_형태소분석.xlsx')